# 붓꽃 데이터 살펴보기 (`Iris1.csv`)

기계학습으로 문제를 풀기 전에, **데이터를 먼저 들여다보는 습관**이 필요합니다.  
오늘은 pandas로 `Iris1.csv`를 열고, **편향적으로 수집·저장되어 있지 않은지** 살펴본 뒤 어떻게 처리할지 생각해 봅시다.

| 오늘 할 일 | 왜 필요한가? |
|---|---|
| 파일을 열어 종류 개수와 저장 순서를 확인하기 | 치우친 데이터로 학습하면 모델도 치우칩니다 |

이 자료는 `2-1-2_데이터_수집.ipynb`에 이어, 표를 직접 열어 편향을 확인합니다.  
열 속성·결측치는 `2-1-3_데이터_전처리.ipynb`에서 다룹니다.

**준비물:** 이 노트북과 같은 폴더에 `Iris1.csv`가 있어야 합니다.  
**실행 방법:** 각 코드 칸의 ▶(실행)을 **위에서부터 차례대로** 누르세요.


## 0. pandas 불러오기와 파일 열기

`pandas`는 표 형태 데이터를 다루는 파이썬 라이브러리입니다.  
한글 CSV는 저장 방식에 따라 글자가 깨질 수 있어서, 교과서처럼 **`encoding='cp949'`** 를 사용합니다.


In [ ]:
import pandas as pd

# 그래프 한글이 깨질 때만 아래 주석을 해제하세요.
# (윈도우) import matplotlib.pyplot as plt; plt.rc('font', family='Malgun Gothic'); plt.rcParams['axes.unicode_minus'] = False
# (코랩)   !pip install koreanize-matplotlib  다음 칸에서 import koreanize_matplotlib

df = pd.read_csv('Iris1.csv', encoding='cp949')
df.head()


> **코랩에서 파일을 직접 올릴 때**는 아래를 사용하세요.
> ```python
> from google.colab import files
> files.upload()   # Iris1.csv 선택
> df = pd.read_csv('Iris1.csv', encoding='cp949')
> ```


In [ ]:
# 끝부분 5행도 꼭 봅니다. 앞만 보면 데이터의 전체 모습을 놓치기 쉽습니다.
df.tail()


In [ ]:
print('행 개수(데이터 건수):', df.shape[0])
print('열 개수(속성 개수):', df.shape[1])
df.shape   # (행, 열)


---

# 1. 편향적으로 수집된 부분이 있을까?

**데이터 편향성**이란, 데이터가 현실을 충분히 반영하지 못하거나 **한쪽으로 치우친 것**을 말합니다.

교과서에서 편향이 생기는 지점은 세 곳입니다.

1. **데이터 수집** — 특정 집단이 너무 많거나, 너무 적거나, 빠져 있음
2. **데이터 전처리** — 잘못된 입력, 결측치를 대충 채움
3. **속성 선택** — 필요 없는 속성을 넣거나, 중요한 속성을 빠뜨림

지금은 1번, **수집·저장 단계에서 치우침이 보이는지**부터 코드로 확인합니다.


### 1-1. 종류별 개수는 고른가?

분류 문제에서는 **정답(종류)이 한쪽으로 몰려 있으면** 모델이 많은 쪽만 잘 맞히고, 적은 쪽은 거의 배우지 못합니다.
`value_counts()`로 종류별 개수를 세어 봅시다.


In [ ]:
df['종류'].value_counts()


In [ ]:
(df['종류'].value_counts(normalize=True) * 100).round(1)


In [ ]:
df['종류'].value_counts().plot(
    kind='bar',
    title='종류별 데이터 개수',
    xlabel='종류',
    ylabel='개수',
    rot=0
)


> **생각하기 1**
> 종류별 개수가 어떻게 나왔나요? 한쪽으로 몰려 있나요, 비슷한가요?
> → (여기에 관찰한 숫자를 적으세요)
>
> 개수가 비슷하면 **종류 불균형 편향은 없어 보입니다.**
> 그런데 개수만 같다고 해서, 데이터가 학습에 바로 써도 될 만큼 공정할까요?
> **저장 순서**도 함께 확인해 봅시다.


### 1-2. 데이터가 어떤 순서로 모여 있는가?

수집하거나 저장할 때 **같은 종류끼리 이어서 붙여 두면**, 나중에 앞부분만 학습용으로 잘라 쓸 때 문제가 생깁니다.
앞·중간·끝의 `종류`를 비교해 보세요.


In [ ]:
print('앞부분 5개')
display(df[['일련번호', '종류']].head())

print('50번째 근처 (48~52행)')
display(df[['일련번호', '종류']].iloc[47:52])

print('100번째 근처 (98~102행)')
display(df[['일련번호', '종류']].iloc[97:102])

print('끝부분 5개')
display(df[['일련번호', '종류']].tail())


In [ ]:
# 종류가 바뀌는 지점을 찾아 보면, 데이터가 어떻게 묶여 있는지 더 분명해집니다.
df.loc[df['종류'] != df['종류'].shift(), ['일련번호', '종류']]


> **생각하기 2**
> 종류가 파일 안에서 어떻게 배열되어 있나요?
> → (예: 처음부터 끝까지 섞여 있다 / 종류별로 덩어리져 있다)

만약 이 표를 **섞지 않고 앞 80%만 학습, 뒤 20%만 시험**에 쓰면 어떤 일이 생길까요? 직접 잘라 봅시다.


In [ ]:
n = int(len(df) * 0.8)   # 전체의 80%
print('학습용으로 앞', n, '행을 가져간다고 가정')
print('\n[학습용] 종류별 개수')
print(df.iloc[:n]['종류'].value_counts())
print('\n[시험용] 종류별 개수')
print(df.iloc[n:]['종류'].value_counts())


> **생각하기 3**
> 시험용 데이터에는 어떤 종류가 들어가나요? 학습용에는 어떤 종류가 부족한가요?
> →
>
> 이런 상태로 모델을 만들면, 시험 점수가 실제 실력보다 **너무 낮거나 이상하게** 나올 수 있습니다.
> 개수는 50개씩으로 고른데, **저장 순서가 종류별로 붙어 있는 것** 자체가 학습을 왜곡하는 편향이 됩니다.


### 1-3. 그렇다면 어떻게 처리해야 할까?

지금 발견한 문제에 쓸 수 있는 처리입니다. **정답은 하나가 아닙니다.** 상황마다 고릅니다.

| 발견한 문제 | 처리할 수 있는 방법 | 언제 쓰면 좋은가? |
|---|---|---|
| 종류가 순서대로 붙어 있음 | 행을 **무작위로 섞기** | 가장 먼저, 가장 쉽게 할 수 있는 조치 |
| 학습/시험에 종류가 고르게 들어가게 하고 싶음 | 종류별로 같은 비율로 나누기(**층화 분할**) | 분류 문제를 본격적으로 학습할 때 |
| 어떤 종류가 실제로 너무 적음 | 그 종류를 **더 수집**하기 | 가장 바람직한 해결. 데이터가 현실을 더 잘 반영함 |
| 더 수집하기 어려움 | 많은 쪽에서 일부를 줄이거나(언더샘플링), 적은 쪽을 늘림(오버샘플링) | 임시 조치. 가짜로 늘리면 과적합 위험이 있음 |

지금은 바로 실행해 볼 수 있는 **섞기**부터 해 봅시다.


In [ ]:
# random_state=7 을 주면, 실행할 때마다 같은 순서로 섞입니다.
df_섞음 = df.sample(frac=1, random_state=7).reset_index(drop=True)

print('섞은 뒤 앞부분')
display(df_섞음[['일련번호', '종류']].head(10))

n = int(len(df_섞음) * 0.8)
print('섞은 다음 앞 80% / 뒤 20%의 종류별 개수')
print('\n[학습용]')
print(df_섞음.iloc[:n]['종류'].value_counts())
print('\n[시험용]')
print(df_섞음.iloc[n:]['종류'].value_counts())


> **생각하기 4**
> 섞기 전과 비교해 보세요. 시험용에 세 종류가 모두 들어가나요?
> →
>
> 섞어도 비율이 완전히 같아지지는 않을 수 있습니다. 나중에 기계학습에서는
> `train_test_split(..., stratify=정답열)` 처럼 **종류 비율을 유지하며 나누는 방법**을 씁니다.


### 1-4. (비교) 만약 한 종류가 훨씬 적게 수집되었다면?

`Iris1.csv` 자체는 종류별로 50개씩입니다.
이번에는 **버지니카를 10개만 모은 상황**을 만들어서, 불균형이 눈에 어떻게 보이는지 비교해 봅시다.


In [ ]:
df_치우침 = pd.concat([
    df[df['종류'] == '세토사'],
    df[df['종류'] == '버시컬러'],
    df[df['종류'] == '버지니카'].head(10)
], ignore_index=True)

print('가상으로 만든 치우친 데이터')
print(df_치우침['종류'].value_counts())
print()
print('비율(%)')
print((df_치우침['종류'].value_counts(normalize=True) * 100).round(1))

df_치우침['종류'].value_counts().plot(
    kind='bar',
    title='(가상) 버지니카가 적게 수집된 경우',
    xlabel='종류',
    ylabel='개수',
    rot=0
)


> **생각하기 5**
> 이 치우친 데이터로 '꽃 종류 맞히기' 모델을 만들면, 어떤 종류를 잘 틀릴 것 같나요? 이유는?
> →
>
> 가장 좋은 해결은 무엇일까요? (더 모으기 / 많은 쪽을 줄이기 / 적은 쪽을 복제해서 늘리기 중에서, 이유를 들어 고르세요)
> →
>
> **한 걸음 더:** 종류 개수가 같아도, 한 지역·한 계절·한 사람의 측정만 모이면 현실의 모든 붓꽃을 대표하지 못합니다.
> 숫자 균형 말고도 '이 데이터가 현실을 충분히 담았는가?'를 질문하는 것이 편향을 보는 태도입니다.


---

# 오늘 확인한 것 정리

코드를 다시 보지 말고, 실행해서 본 기억으로 채우세요.

| 확인 항목 | 내가 본 결과 | 그래서 어떻게 하면 좋은가? |
|---|---|---|
| 종류별 개수 |  |  |
| 데이터가 저장된 순서 |  |  |
| 섞지 않고 앞 80%만 학습하면? |  |  |

**자기 점검**

- [ ] `read_csv`로 파일을 열고 `head()`, `shape`로 전체 크기를 확인했다.
- [ ] `value_counts()`로 종류 개수를 확인했다.
- [ ] 저장 순서 때문에 앞뒤 분할이 위험한 이유를 말할 수 있다.
- [ ] 치우침을 줄이기 위한 처리(섞기, 더 수집하기 등)를 말할 수 있다.

다음 시간에는 `2-1-3_데이터_전처리.ipynb`에서 **열 속성**과 **결측치·이상치**를 다룹니다.
